---
title: Get started with Q-CTRL Dynamics Simulator
description: Load the Q-CTRL Dynamics Simulator function, calculate the function size, and simulate a four-site Fermi-Hubbard chain.
---

{/* cspell:ignore eeeu */}

# Get started with Q-CTRL Dynamics Simulator

*See the [overview](/docs/functions/q-ctrl-dynamics-simulator) and the [API reference](/docs/functions/q-ctrl-dynamics-simulator/api-reference)*

<Admonition type="note">
   IBM Quantum&reg; Functions are a beta feature available only to IBM Quantum Premium Plan, Flex Plan, and On-Prem (through the IBM Quantum Platform API) Plan users. Beta features are subject to change.
</Admonition>

This guide uses a small one-dimensional Fermi-Hubbard example to show the complete workflow from defining the model to plotting the measured dynamics.

## Set up and load the function

Find access information for this quantum function in the [Quantum Functions Catalog](https://quantum.cloud.ibm.com/functions).

This guide shows how to load the Dynamics Simulator and simulate a four-site Fermi-Hubbard chain. The following code assumes that you already saved your IBM Quantum Platform API key and instance CRN to your local environment.

For account setup and client installation instructions, see [Set up Functions](/docs/ibm-quantum-platform/functions-get-started).

In [ ]:
from qiskit_ibm_catalog import FunctionsCatalog

catalog = FunctionsCatalog(channel="ibm_quantum_platform")

In [2]:
# Access the Q-CTRL Dynamics Simulator quantum function.
sim_qf = catalog.load("q-ctrl/dynamics-simulator")

In [3]:
# Check the list of backends you have access to
catalog.backends()

[<IBMBackend('ibm_boston')>,
 <IBMBackend('ibm_fez')>,
 <IBMBackend('ibm_kingston')>,
 <IBMBackend('ibm_marrakesh')>,
 <IBMBackend('ibm_miami')>,
 <IBMBackend('ibm_phoenix')>,
 <IBMBackend('ibm_pittsburgh')>]

In [4]:
# Select the backend you want to use
backend = catalog.backend("ibm_phoenix")

## Function sizes

Every job runs at one size: `S`, `M`, or `L`. If the size is too small, the job is rejected. If it is larger than needed, the job runs on a bigger compute profile than the work requires. See [Function sizes](/docs/managed-workflows/managed-workflow-compute-profiles#best-practice-match-sizes-to-problem-sizes) for how sizes work in general.

Pass the size to `sim_qf.run()` with the `function_size` argument. For the Dynamics Simulator, the size depends on the number of qubits, Trotter steps, and observables.

Each lattice site requires two qubits. The number of Trotter steps is set by `simulation["step_count"]`, and the number of observables is the number of entries in `observables`.

| Size | Qubits | Trotter steps | Observables | Rule |
| --- | --- | --- | --- | --- |
| `S` | ≤ 20 | ≤ 10 | ≤ 20 | All three metrics must satisfy the Small limits. |
| `M` | ≤ 50 | ≤ 20 | ≤ 50 | At least one metric exceeds its Small limit, but all three remain within the Medium limits. |
| `L` | > 50 | > 20 | > 50 | Any single metric exceeds its Medium limit. |

Select the size required by the largest of the three requirements. For example, a simulation with 20 qubits, 30 Trotter steps, and 20 observables requires `"L"` because the number of Trotter steps exceeds the Medium limit.

### Calculate the size before submitting

Use the following helper to calculate the size from the number of qubits, Trotter steps, and observables. It returns the smallest size that satisfies all three requirements. No job is submitted.

In [4]:
def calculate_function_size(num_qubits, num_steps, num_observables):
    if num_qubits <= 20 and num_steps <= 10 and num_observables <= 20:
        return "S"
    if num_qubits <= 50 and num_steps <= 20 and num_observables <= 50:
        return "M"
    return "L"

If the size is too small, the job fails with error `4600`, and the message names the size to use instead. If it is larger than needed, the job runs normally and records a `4601` advisory event. See [Function sizes](/docs/managed-workflows/managed-workflow-compute-profiles#best-practice-match-sizes-to-problem-sizes) to learn more.

## Example: Simulate a four-site Fermi-Hubbard chain

Consider a four-site one-dimensional Fermi-Hubbard chain with hopping amplitude $t_h=1$, no on-site interaction, and zero chemical potential. A single spin-up fermion begins at site 0 and evolves across the chain.

This small problem keeps the focus on the Dynamics Simulator workflow:

1. Define the model and simulation.
2. Define the initial state and observables.
3. Submit the quantum function job.
4. Retrieve the expectation values.
5. Plot the dynamics.

### 1. Define the problem

In [5]:
site_count = 4

model = {
    "hopping_amplitude": 1.0,
    "on_site_interaction": 0.0,
    "chemical_potential": 0.0,
}

simulation = {
    "time_step": 0.2,
    "step_count": 8,
}

initial_state = "eeeu"

observables = [
    {
        "operators": [[["Nup", site]]],
        "coefficients": [1.0],
    }
    for site in range(site_count)
]

shot_count = 2048

The observable list measures the spin-up occupation

$$
\langle n_{i\uparrow} \rangle
$$

at each site. The observables are returned in the same order in which they are supplied.

### 2. Run the dynamics simulation

Submit the problem through the quantum function. The Quantum Functions Catalog handles IBM Quantum authentication, so you don't pass separate Fire Opal or hardware-provider credentials to `sim_qf.run()`. Calculate the function size from the simulation inputs, as described in [Function sizes](#function-sizes), and pass the returned value through `function_size`. Each lattice site in `initial_state` requires two qubits, `simulation["step_count"]` gives the number of Trotter steps, and `len(observables)` gives the number of observables.

In [6]:
# Calculate the workload size
function_size = calculate_function_size(
    num_qubits=2 * len(initial_state),
    num_steps=simulation["step_count"],
    num_observables=len(observables),
)
print(f"Workload size: {function_size}")

Workload size: S


In [6]:
job = sim_qf.run(
    model=model,
    simulation=simulation,
    observables=observables,
    initial_state=initial_state,
    shots=shot_count,
    backend_name=backend.name,
    function_size=function_size,
)

In [9]:
print(job.job_id, job.status())

68202fad-6919-425d-8fd2-c1c649cd15b1 DONE


The returned object is a quantum function job. You can keep the job ID to retrieve or inspect the workload later.

### 3. Retrieve the result

In [10]:
result = job.result()

print("Result keys:", sorted(result.keys()))
print("Warnings:", result.get("warnings", []))

Result keys: ['metadata', 'observable_expectation_values', 'observables', 'warnings']
Warnings: []


Convert the observable expectation values to a NumPy array. Each row corresponds to one observable, and each column corresponds to a recorded simulation time, beginning at $t=0$. For this example, the four rows contain the spin-up occupations of sites 0 through 3.

In [11]:
import numpy as np

expectation_values = np.asarray(
    result["observable_expectation_values"],
    dtype=float,
)

print("Expectation-value array shape:", expectation_values.shape)

Expectation-value array shape: (4, 9)


### 4. Visualize the dynamics

The initial state contains one spin-up fermion and no spin-down fermions. Its evolution therefore remains in the single-particle subspace, whose four basis states correspond to the fermion occupying sites 0, 1, 2, and 3.

In this subspace, the Hamiltonian connects neighboring sites with hopping amplitude $-t_h$. For the parameters used here,

$$
H_{\mathrm{1p}}
=
-t_h
\begin{pmatrix}
0 & 1 & 0 & 0 \\
1 & 0 & 1 & 0 \\
0 & 1 & 0 & 1 \\
0 & 0 & 1 & 0
\end{pmatrix}.
$$

Diagonalize this matrix with NumPy and evolve the initial state at the same times that were returned by the Dynamics Simulator. The squared magnitude of each state amplitude gives the spin-up occupation of the corresponding site.

This calculation gives exact Hamiltonian evolution. Differences between the hardware estimates and this reference can include the finite Trotter step, measurement statistics, and residual hardware errors.

In [12]:
times = np.arange(expectation_values.shape[1]) * simulation["time_step"]

# This reference is specific to a single spin-up fermion.
if (
    len(initial_state) != site_count
    or initial_state.count("u") != 1
    or initial_state.count("e") != site_count - 1
):
    raise ValueError(
        "This reference requires one spin-up fermion and otherwise empty sites."
    )

# Build the Hamiltonian in the single-particle subspace.
hamiltonian = -model["hopping_amplitude"] * (
    np.eye(site_count, k=1) + np.eye(site_count, k=-1)
)
hamiltonian -= model["chemical_potential"] * np.eye(site_count)

# The rightmost character of initial_state corresponds to site 0.
initial_amplitudes = np.zeros(site_count, dtype=complex)
initial_site = initial_state[::-1].index("u")
initial_amplitudes[initial_site] = 1.0

# Evolve the initial state in the Hamiltonian eigenbasis.
energies, eigenvectors = np.linalg.eigh(hamiltonian)
initial_coefficients = eigenvectors.conj().T @ initial_amplitudes

phases = np.exp(-1j * np.outer(energies, times))
states = eigenvectors @ (initial_coefficients[:, None] * phases)

ideal_expectation_values = np.abs(states) ** 2

Plot one panel for each site. The solid curves show the exact reference at the recorded times, and the markers show the hardware estimates returned by the Dynamics Simulator.

In [19]:
import matplotlib.pyplot as plt
import qctrlvisualizer as qv
from qctrlvisualizer import QCTRL_STYLE_COLORS as colors

plt.style.use(qv.get_qctrl_style())

fig, ax = plt.subplots(figsize=(10, 6))

for site in range(len(expectation_values)):
    site_color = colors[site % len(colors)]

    ax.plot(
        times,
        ideal_expectation_values[site],
        color=site_color,
        linewidth=2,
        label=f"Site {site}: exact reference",
    )

    ax.plot(
        times,
        expectation_values[site],
        color=site_color,
        linestyle="--",
        linewidth=1.5,
        marker="o",
        markersize=5,
        label=f"Site {site}: hardware estimate",
    )

ax.set_xlim(times[0], times[-1])
ax.set_ylim(-0.05, 1.05)

ax.set_xlabel(r"Time (units of $\hbar/t_h$)")
ax.set_ylabel(r"$\langle n_{i\uparrow} \rangle$")
ax.set_title("Spin-up occupation dynamics")

ax.legend(
    ncol=2,
    fontsize=10,
    loc="best",
)

fig.tight_layout()
plt.show()

<Image src="/docs/images/guides/q-ctrl-dynamics-simulator/extracted-outputs/61a7ad6e-0.avif" alt="Output of the previous code cell" />

At $t=0$, the exact spin-up occupation is 1 at site 0 and 0 at the other sites. As the system evolves, hopping spreads the particle's probability across the chain. The occupation of site 1 increases first, followed by increasing occupation at sites 2 and 3. For the parameters in this example, the final recorded time is $t=1.6$. The exact occupations of sites 0 through 3 are approximately 0.026, 0.370, 0.392, and 0.211, respectively.

The exact occupations satisfy the following equation at every time, reflecting conservation of particle number:

$$
\sum_{i=0}^{3}\langle n_{i\uparrow}(t)\rangle=1
$$

Individual site occupations can rise and fall as the particle evolves through the finite chain.

## Fetch error messages

If the workload enters the `ERROR` state, call `job.result()` to retrieve the error message. You can also inspect the function logs, if they are available.

In [20]:
print(job.status())

if job.status() == "ERROR":
    print(job.result())
    print(job.logs())

DONE


Common input-validation issues include an unsupported model or simulation method, an initial-state string that is too long, mismatched observable terms and coefficients, or an observable that references a site outside the initial-state string.

## Next steps

<Admonition type="tip" title="Recommendations">

- Try the [Spin and charge dynamics in the Fermi-Hubbard model](/docs/tutorials/fermi-hubbard-spin-charge-dynamics) tutorial.
- Visit the [API reference](/docs/functions/q-ctrl-dynamics-simulator/api-reference) for this quantum function.
- Modify the model parameters, initial state, time step, step count, and observables to study a different one-dimensional Fermi-Hubbard dynamics problem.
- Learn how to choose [problem sizes and compute profiles](/docs/managed-workflows/managed-workflow-compute-profiles).

</Admonition>